In [ ]:
library(edgeR)
library(Matrix)
library(dplyr)
library(tibble)

indir <- "/path/to/nature_study/RNA_export_for_stats"
outdir <- "/path/to/nature_study/RNA_pseudobulk_edgeR"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

# =========================
# load exported pseudobulk
# =========================
pb_mat <- readRDS(file.path(indir, "HSC_MPP_pseudobulk_counts.rds"))
pb_meta <- read.delim(file.path(indir, "HSC_MPP_pseudobulk_coldata.tsv"), check.names = FALSE)

pb_meta$condition <- factor(pb_meta$condition, levels = c("disomy", "Ts21"))
pb_meta <- pb_meta %>% arrange(condition, donor)

pb_mat <- pb_mat[, pb_meta$group_id, drop = FALSE]

stopifnot(all(colnames(pb_mat) == pb_meta$group_id))

print(dim(pb_mat))
print(pb_meta)
print(table(pb_meta$condition))

# =========================
# edgeR pseudobulk DE
# =========================
y <- DGEList(counts = pb_mat, samples = pb_meta)

design <- model.matrix(~ condition, data = pb_meta)
rownames(design) <- pb_meta$group_id

keep <- filterByExpr(y, design = design)
table(keep)

y <- y[keep, , keep.lib.sizes = FALSE]
y <- calcNormFactors(y)
y <- estimateDisp(y, design)
fit <- glmQLFit(y, design, robust = TRUE)
qlf <- glmQLFTest(fit, coef = "conditionTs21")

res <- topTags(qlf, n = Inf)$table %>%
  rownames_to_column("gene") %>%
  mutate(
    direction = case_when(
      logFC > 0 ~ "higher_in_Ts21",
      logFC < 0 ~ "higher_in_disomy",
      TRUE ~ "no_change"
    )
  ) %>%
  arrange(FDR, desc(abs(logFC)))

write.table(
  res,
  file = file.path(outdir, "HSC_MPP_pseudobulk_edgeR_all_genes.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

saveRDS(
  list(y = y, design = design, fit = fit, qlf = qlf, results = res),
  file = file.path(outdir, "HSC_MPP_pseudobulk_edgeR_results.rds")
)

# =========================
# summaries
# =========================
cat("Genes before filtering:", nrow(pb_mat), "\n")
cat("Genes tested after filterByExpr:", nrow(y), "\n")
cat("FDR <= 0.05:", sum(res$FDR <= 0.05, na.rm = TRUE), "\n")
cat("Higher in Ts21:", sum(res$FDR <= 0.05 & res$logFC > 0, na.rm = TRUE), "\n")
cat("Higher in disomy:", sum(res$FDR <= 0.05 & res$logFC < 0, na.rm = TRUE), "\n")

head(res, 20)

# =========================
# B-lineage panel output
# =========================
gene_panel <- c("EBF1", "PAX5", "CD24", "DNTT", "BCL11A", "SPI1", "SPIB", "IRF4", "IRF8", "TCF3", "TCF12")

panel_res <- res %>%
  filter(gene %in% gene_panel) %>%
  arrange(match(gene, gene_panel))

write.table(
  panel_res,
  file = file.path(outdir, "HSC_MPP_pseudobulk_edgeR_Blineage_panel.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

panel_res

In [ ]:
table(pb_meta$condition)
sum(res$FDR <= 0.05, na.rm = TRUE)
panel_res

In [ ]:
genes_show <- c(
  # core B-lineage / lymphoid priming
  "EBF1", "PAX5", "CD24", "DNTT",
  "BCL11A", "SPI1", "SPIB", "IRF4", "IRF8", "TCF3", "TCF12",
  "IKZF1", "IKZF3", "IL7R", "FLT3",

  # early B-cell program
  "VPREB1", "IGLL1", "RAG1", "RAG2",
  "CD79A", "CD79B", "BLNK", "POU2AF1", "ETS1",

  # useful comparator / stem-lymphoid context
  "MEF2C", "RUNX1", "BACH2", "LEF1", "HHEX"
)



In [ ]:
genes_show <- intersect(genes_show, res$gene)

panel_res <- res[res$gene %in% genes_show, , drop = FALSE]

write.table(
  panel_res,
  file = file.path(outdir, "HSC_MPP_pseudobulk_edgeR_Blineage_panel.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

In [ ]:
genes_compare <- c(
  # erythroid / heme
  "GATA1", "KLF1", "ALAS2", "HBB", "HBG1", "HBG2", "HBZ",

  # interferon / ISG
  "IFI6", "ISG15", "IFITM1", "IFITM3", "MX1", "IFI44L", "OAS1"
)

genes_compare <- intersect(genes_compare, res$gene)

compare_res <- res[res$gene %in% genes_compare, , drop = FALSE]

write.table(
  compare_res,
  file = file.path(outdir, "HSC_MPP_pseudobulk_edgeR_comparator_panel.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

In [ ]:
# final B-lineage panel
genes_main <- c(
  "EBF1", "PAX5", "RAG1", "IGLL1", "BLNK", "IRF8", "BACH2"
)

outdir <- "/path/to/nature_study/RNA_pseudobulk_edgeR"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

# keep only genes that are actually present in the edgeR results
genes_main_present <- intersect(genes_main, res$gene)

# 1) gene-level stats table
panel_main <- res[match(genes_main_present, res$gene), , drop = FALSE]
panel_main <- panel_main[!is.na(panel_main$gene), , drop = FALSE]

write.table(
  panel_main,
  file = file.path(outdir, "HSC_MPP_pseudobulk_edgeR_main7_panel.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

panel_main




In [ ]:
library(edgeR)

# logCPM matrix from the filtered edgeR object
logcpm_mat <- cpm(y, log = TRUE, prior.count = 2)

genes_main_present <- intersect(genes_main, rownames(logcpm_mat))

plot_list_main <- vector("list", length(genes_main_present))

for (i in seq_along(genes_main_present)) {
  g <- genes_main_present[i]
  tmp <- data.frame(
    gene = g,
    group_id = colnames(logcpm_mat),
    logCPM = as.numeric(logcpm_mat[g, ]),
    stringsAsFactors = FALSE
  )
  plot_list_main[[i]] <- tmp
}

plot_df_main <- do.call(rbind, plot_list_main)
plot_df_main <- merge(plot_df_main, pb_meta, by = "group_id", all.x = TRUE, sort = FALSE)

idx <- match(plot_df_main$gene, res$gene)
plot_df_main$logFC <- res$logFC[idx]
plot_df_main$PValue <- res$PValue[idx]
plot_df_main$FDR <- res$FDR[idx]
plot_df_main$direction <- res$direction[idx]

write.table(
  plot_df_main,
  file = file.path(outdir, "HSC_MPP_pseudobulk_edgeR_main7_plot_ready_logCPM.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

head(plot_df_main)